<a id="multi-analyzer-pipelines"></a>
# VideoDB Understanding: Multi-Analyzer Pipelines

Build a dependency graph where downstream analyzers use timestamp-aligned outputs from earlier analyzers.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/multi-analyzer-pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install, connect, and choose a video

In [ ]:
!pip install -q videodb python-dotenv pandas

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")
print("Collection:", collection.id)

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

video = collection.upload(VIDEO_URL)

# To use an existing video instead:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Video:", video.id)
video.play()

<a id="graph"></a>
## 2. Design the graph

This example runs transcript and OCR in parallel. The VLM waits for both and receives their output for the matching scene.

```text
transcript ─┐
            ├──► scene
text (OCR) ─┘
```

Dependencies use analyzer **names**, not types.

## 3. Submit the pipeline

In [ ]:
pipeline = video.understand(
    analyzers=[
        {"type": "spoken_words", "name": "transcript", "config": {"language": "en"}},
        {
            "type": "ocr",
            "name": "text",
            "sampling": {"strategy": "uniform", "frame_count": 3},
            "config": {"model": "ultra"},
        },
        {
            "type": "vlm",
            "name": "scene",
            "inputs": ["transcript", "text"],
            "sampling": {"strategy": "uniform", "frame_count": 4},
            "config": {
                "model": "ultra",
                "prompt_inputs": ["transcript", "text"],
                "prompt": (
                    "Describe the scene using visible evidence. Use spoken words and OCR text "
                    "when they clarify what is happening."
                ),
                "schema": {
                    "scene_description": "text",
                    "spoken_context": "text",
                    "visible_text": ["string"],
                },
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding:", pipeline.id)
for analyzer in pipeline.list_analyzers():
    print(analyzer.name, analyzer.type, analyzer.status)

## 4. Wait and inspect every stage

In [ ]:
import pandas as pd

pipeline.wait_until_complete(timeout=3600, poll_interval=15)

outputs = {
    name: pipeline.get_analyzer(name).get_output()
    for name in ("transcript", "text", "scene")
}

for name, output in outputs.items():
    print(name, len(output.get("scenes", [])), "scenes")

scene_rows = [
    {"start": s.get("start"), "end": s.get("end"), **(s.get("data") or {})}
    for s in outputs["scene"].get("scenes", [])
]
pd.DataFrame(scene_rows).head()

<a id="rules"></a>
## 5. Dependency rules

Requests are validated before a run is created:

- Names must be unique.
- Every `inputs` name must exist.
- An analyzer cannot depend on itself.
- Dependency cycles are rejected.
- `prompt_inputs` must be a subset of `inputs`.
- Speech and object detection produce inputs but do not accept analyzer inputs.

Keep independent analyzers parallel. Add dependencies only when a downstream analyzer needs their data.

## Optional cleanup

In [ ]:
DELETE_PIPELINE = False
if DELETE_PIPELINE:
    pipeline.delete()